# 対策の比較

再推論を止めるタイミングを 3 条件で試し、対策なしと比較する。

対策の動作は次のとおりである（`scripts/rollout.py` の `next_interval`）。

- 推論のたびに、チャンクの予定軌道の最下点までの距離を求める。対象物の位置は使わない
- 距離が発動距離を下回ったら、アプローチ完了から少し先まで再推論を止め、そのチャンクを実行する
- 発動距離を次の間隔の途中で横切る場合は、推論を前倒しする。距離帯の悪い計画を凍結しないため
- 上昇中、グリッパーを閉じた後、横移動が主の軌道では発動しない。1 エピソードに 1 回だけ発動する

対象は、誤差の大きい上位 100 本とランダムな 100 本である。同じタスク、エピソード、シードで実行する。

実行前に `/content/rclone.conf`、`/content/hf_token`、`/content/rollout.py` をアップロードしておく。

## 1. セットアップ

In [ ]:
# VM の cmake は pip の隔離ビルドで動かないため、システムの cmake を先に使う
!apt-get install -y -q cmake > /dev/null
!PATH=/usr/bin:$PATH pip install -q "lerobot[pi,libero]==0.6.1"
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null

In [ ]:
import importlib.util
import json
import os
import pathlib
import subprocess
import sys
import time

import yaml
from huggingface_hub import login, snapshot_download

RCLONE = ["--config", "/content/rclone.conf", "--retries", "10"]
REMOTE_DIR = os.environ.get("RCLONE_REMOTE_ROOT", "remote:pai_homework2")  # rclone の保存先

login(token=pathlib.Path("/content/hf_token").read_text().strip())

# ヘッドレス環境で MuJoCo を GPU 描画する
os.environ["MUJOCO_GL"] = "egl"
icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

# libero は設定ファイルがないと import 時に input() で止まるため、先に作っておく
libero_root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
config_file = pathlib.Path.home() / ".libero/config.yaml"
config_file.parent.mkdir(exist_ok=True)
config_file.write_text(yaml.dump({
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}))

# 対策なしの収集結果を取得する
subprocess.run(["rclone", "copyto", f"{REMOTE_DIR}/approach_analysis/results.tar.gz", "/content/results.tar.gz", *RCLONE], check=True)
subprocess.run(["tar", "xzf", "/content/results.tar.gz", "-C", "/content"], check=True)

## 2. 実行

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

sys.path.insert(0, "/content")
from rollout import *
from rollout import approach_end

FREEZE_DEPTHS_M = [0.16, 0.12, 0.08]  # 発動距離の条件
NUM_TOP = 100  # 誤差の大きい順に選ぶ本数
NUM_RANDOM = 100  # ランダムに選ぶ本数
TARGET_FILE = pathlib.Path("/content/atomic/targets.json")
TARGET_FILE.parent.mkdir(exist_ok=True)

problems = pd.read_csv(LOCAL_DIR / "problem_rollouts.csv")
top = [(r.suite, r.task_id, r.episode) for r in problems.head(NUM_TOP).itertuples()]
rest = [rollout for rollout in all_rollouts() if rollout not in set(top)]
picked = np.random.default_rng(SEED).permutation(len(rest))[:NUM_RANDOM]
random_rollouts = [rest[i] for i in picked]
TARGET_FILE.write_text(json.dumps(top + random_rollouts))


def sync_to_drive():
    """対策の結果をアーカイブにまとめて Drive に送る。"""
    subprocess.run(["tar", "czf", "/content/atomic.tar.gz", "-C", "/content", "atomic"], check=True)
    subprocess.run(["rclone", "copyto", "/content/atomic.tar.gz", f"{REMOTE_DIR}/approach_analysis/atomic.tar.gz", *RCLONE], check=True)


# ワーカーが同時にダウンロードしないよう、モデルとアセットを先に取得する
snapshot_download(MODEL_ID)
AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
make_env(SUITES[0], 0)[0].close()

for depth in FREEZE_DEPTHS_M:
    atomic_dir(depth).mkdir(parents=True, exist_ok=True)
    logs = [open(atomic_dir(depth) / f"worker{i}.log", "w") for i in range(NUM_WORKERS)]
    workers = [
        subprocess.Popen([sys.executable, "/content/rollout.py", str(i), str(depth)], stdout=log, stderr=subprocess.STDOUT)
        for i, log in enumerate(logs)
    ]
    while any(worker.poll() is None for worker in workers):
        time.sleep(30)
    sync_to_drive()
    print(f"{depth * 100:.0f}cm 完了: {len(list(atomic_dir(depth).glob('*.npz')))} 本", flush=True)
    assert all(worker.returncode == 0 for worker in workers), "worker のログを確認する"

## 3. 比較

対策なしの全データから求めた把持オフセットを基準に、次の値を比べる。

| 列 | 内容 |
|---|---|
| `success` | タスクの成功率 |
| `triggered` | 対策が発動した割合 |
| `freeze_h` | 発動時の高さの差 [cm] |
| `end_gap_abs` | アプローチ完了時の、基準の高さからのずれの絶対値 [cm] |
| `stall` | 下降指令中に高さが動かなかったステップ数 |
| `steps` | タスクの総ステップ数 |

`stall` は、z 指令が -0.2 より強いのに高さの変化が 1mm 未満のステップを、アプローチ完了まで数えた値である。

In [ ]:
def load(path):
    with np.load(path) as npz:
        return {key: npz[key] for key in npz.files}


def task_offsets():
    """対策なしの全データから、タスクごとの把持オフセットを求める。"""
    rows = []
    for suite_name, task_id, episode in all_rollouts():
        data = load(episode_path(suite_name, task_id, episode))
        end = approach_end(data["step_action"])
        if end is not None:
            relative = data["step_eef"][end] - data["step_obj"][end]
            if np.linalg.norm(relative[:2]) <= REACH_RADIUS_M and relative[2] <= REACH_HEIGHT_M:
                rows.append({"key": (suite_name, task_id), "offset": relative[2]})
    return pd.DataFrame(rows).groupby("key")["offset"].median().to_dict()


def measure(data, offset):
    """1 ロールアウトの指標を返す。"""
    actions = data["step_action"]
    heights = data["step_eef"][:, 2] - data["step_obj"][:, 2]
    end = approach_end(actions)
    freeze_t = int(data["freeze_t"]) if "freeze_t" in data else -1
    row = {
        "success": bool(data["success"]), "steps": len(actions), "triggered": freeze_t >= 0,
        "freeze_t": freeze_t, "freeze_h": (heights[freeze_t] - offset) * 100 if freeze_t >= 0 else np.nan,
        "end_gap_abs": np.nan, "stall": np.nan,
    }
    if end is not None:
        drops = np.diff(data["step_eef"][:end + 1, 2])
        row["end_gap_abs"] = abs(heights[end] - offset) * 100
        row["stall"] = int(((actions[:end, 2] < -0.2) & (drops > -0.001)).sum())
    return row


offsets = task_offsets()
top_set = set(top)
rows = []
for depth in [None, *FREEZE_DEPTHS_M]:
    for rollout in top + random_rollouts:
        directory = EPISODE_DIR if depth is None else atomic_dir(depth)
        row = measure(load(episode_path(*rollout, directory)), offsets[rollout[:2]])
        rows.append(row | {"arm": "none" if depth is None else f"{depth * 100:.0f}cm",
                           "group": "top" if rollout in top_set else "random",
                           "suite": rollout[0], "task_id": rollout[1], "episode": rollout[2]})

results = pd.DataFrame(rows)
results.to_csv("/content/atomic/compare.csv", index=False)
columns = ["success", "triggered", "freeze_h", "end_gap_abs", "stall", "steps"]
summary = results.groupby(["group", "arm"])[columns].mean().round(3)
print(summary.to_string())
print(results.groupby(["group", "arm"])["end_gap_abs"].quantile(0.9).round(2).unstack().to_string())

# 動画用に、対策の発動点を一覧にする。問題ロールアウトの順に並べる
for depth in FREEZE_DEPTHS_M:
    video_rows = []
    for rollout in top:
        data = load(episode_path(*rollout, atomic_dir(depth)))
        if int(data["freeze_t"]) < 0:
            continue
        t = int(data["freeze_t"])
        chunk = data["chunks"][list(data["infer_t"]).index(t)]
        height = data["step_eef"][t, 2] - data["step_obj"][t, 2]
        video_rows.append({
            "suite": rollout[0], "task_id": rollout[1], "episode": rollout[2], "t": t,
            "required": height - offsets[rollout[:2]],
            "planned": -FREE_GAIN * np.clip(chunk[:approach_end(chunk), 2], -1, 1).sum(),
        })
    pd.DataFrame(video_rows).to_csv(atomic_dir(depth) / "video_rollouts.csv", index=False)

sync_to_drive()

## 4. 後始末

認証情報を VM から削除する。VM の停止は手元から `colab stop` で行う。

In [ ]:
!shred -u /content/rclone.conf /content/hf_token